### managed-agent02 — Missing Dimension Check with Managed Agents

A rewrite of [02-diagram_check01.ipynb](02-diagram_check01.ipynb) (compare the baseline drawing `data/sample01.png` against the check drawing `data/check01.png`, and annotate missing dimensions on the check drawing with red circles), reimplemented with the **Managed Agents API** (`client.beta.agents` / `environments` / `sessions`).

It uses the same public `Anthropic` client (`CLAUDE_API_KEY` / `MODEL`) as `managed-agent01.ipynb`.

### Differences from 02
- **No multi-turn / checkpoint loop needed.** Managed Agents runs the sandbox and the agent loop server-side, working autonomously until the `session` becomes `idle`.
- Images are uploaded via the Files API and **mounted into the container as session `resources`** (the agent reads and processes them as files).
- The skill is **attached to the agent via `skills`** (referenced by `skill_id`).

### Flow
1. Create the skill (`missing-dimension-check`)
2. Create the agent (skill + built-in toolset)
3. Create the environment (cloud / Pillow)
4. Upload the images (sample01 / check01)
5. Create the session (mount the images)
6. Run (streaming) and retrieve the result image

In [ ]:
import os
from anthropic import Anthropic
from anthropic.lib import files_from_dir
from dotenv import load_dotenv
load_dotenv(override=True)

api_key = os.getenv("CLAUDE_API_KEY")
deployment_name = os.getenv("MODEL")

client = Anthropic(api_key=api_key)
print(f"Model: {deployment_name}")

### 1. Create the skill (missing-dimension-check)

Register `.claude/skills/missing-dimension-check`. If a custom skill with the same title already exists, reuse it.

In [ ]:
SKILL_TITLE = "missing-dimension-check"
SKILL_DIR = ".claude/skills/missing-dimension-check"

# Reuse the custom skill with the same title if it exists, otherwise create it
skill = None
for s in client.beta.skills.list(source="custom", betas=["skills-2025-10-02"]).data:
    if s.display_title == SKILL_TITLE:
        skill = s
        print(f"Reusing existing skill: {skill.id}")
        break

if skill is None:
    skill = client.beta.skills.create(
        display_title=SKILL_TITLE,
        files=files_from_dir(SKILL_DIR),
        betas=["skills-2025-10-02"],
    )
    print(f"Created skill: {skill.id}")

print(f"latest_version: {getattr(skill, 'latest_version', 'N/A')}")

### 2. Create the agent

Attach the skill via `skills`, and enable the built-in toolset (`agent_toolset_20260401`: Read / Write / Edit / Bash, etc.).

In [ ]:
agent = client.beta.agents.create(
    name="dimension-check-agent",
    model=deployment_name,
    system=(
        "You are an agent that inspects engineering drawings for missing dimensions. "
        "Work strictly according to the procedure in the attached missing-dimension-check skill. "
        "Do not fabricate numbers. Always make the final judgment visually, and respond in English."
    ),
    skills=[{"type": "custom", "skill_id": skill.id, "version": "latest"}],
    tools=[{"type": "agent_toolset_20260401"}],
)

print(f"Agent ID: {agent.id}, version: {agent.version}")

### 3. Create the environment

Create the cloud sandbox where the agent runs. Since the skill uses PIL (Pillow), install it explicitly via `packages`.

In [ ]:
environment = client.beta.environments.create(
    name="dimension-check-env",
    config={
        "type": "cloud",
        "networking": {"type": "unrestricted"},
        # The skill uses PIL (Pillow) for image processing, so install it explicitly
        "packages": {"type": "packages", "pip": ["pillow"]},
    },
)

print(f"Environment ID: {environment.id}")

In [ ]:
# Use the existing environment ID directly
ENVIRONMENT_ID = "env_01LAMzJCDJbX7D2HFhZJjU44"

# Retrieve the environment to verify it exists
environment = client.beta.environments.retrieve(ENVIRONMENT_ID)
print(f"Environment ID: {environment.id}")

### 4. Upload the images

Upload the baseline drawing `data/sample01.png` and the check drawing `data/check01.png` to the Files API. To preserve the legibility of the dimension text, upload them at original size without downscaling (the agent crops and zooms into regions inside the sandbox).

In [ ]:
uploaded1 = client.beta.files.upload(
    file=("sample01.png", open("data/sample01_resized.png", "rb"), "image/png"),
)
uploaded2 = client.beta.files.upload(
    file=("check01.png", open("data/check01_resized.png", "rb"), "image/png"),
)

print(f"sample01.png (baseline drawing) -> {uploaded1.id}")
print(f"check01.png  (check drawing)    -> {uploaded2.id}")

In [ ]:
# Use the already-uploaded file IDs directly
uploaded1_id = "file_011CcHi4hibynrehA8Wmrh4c"  # sample01.png (baseline drawing)
uploaded2_id = "file_011CcHi4jncQUzKYipqmRfzG"  # check01.png  (check drawing)

print(f"sample01.png (baseline drawing) -> {uploaded1_id}")
print(f"check01.png  (check drawing)    -> {uploaded2_id}")

### 5. Create the session (mount the images)

Mount the two uploaded files into the container as `resources`. The agent reads the files at these paths to do its work.

In [ ]:
MOUNT_SAMPLE = "/mnt/session/uploads/sample01.png"
MOUNT_CHECK = "/mnt/session/uploads/check01.png"

session = client.beta.sessions.create(
    agent=agent.id,
    environment_id=environment.id,
    title="Missing dimension check",
    resources=[
        #{"type": "file", "file_id": uploaded1.id, "mount_path": MOUNT_SAMPLE},
        {"type": "file", "file_id": uploaded1_id, "mount_path": MOUNT_SAMPLE},
        #{"type": "file", "file_id": uploaded2.id, "mount_path": MOUNT_CHECK},
        {"type": "file", "file_id": uploaded2_id, "mount_path": MOUNT_CHECK},
    ],
)

print(f"Session ID: {session.id}")

### 6. Run (streaming)

Send a user message and receive events via streaming. The agent is done when `session.status_idle` is received. Any images contained in `agent.tool_result` during the run (such as the red-circle annotated image) are saved to `out/`.

In [ ]:
with open("./user_prompt_template_01.txt", "r", encoding = 'utf-8') as f:
    user_prompt = f.read()

In [ ]:
import os
import base64

os.makedirs("out", exist_ok=True)


def save_image_block(block, idx):
    """Save an image block contained in a tool_result to out/."""
    src = block.source
    if src.type == "base64":
        ext = src.media_type.split("/")[-1] or "png"
        path = os.path.join("out", f"agent_image_{idx}.{ext}")
        with open(path, "wb") as f:
            f.write(base64.b64decode(src.data))
        print(f"\n\U0001F4BE Saved image: {path}")
    elif src.type == "file":
        fc = client.beta.files.download(file_id=src.file_id, betas=["files-api-2025-04-14"])
        path = os.path.join("out", f"agent_image_{idx}.png")
        fc.write_to_file(path)
        print(f"\n\U0001F4BE Saved image (file_id={src.file_id}): {path}")


instructions = (
    "Two drawings are mounted at the following paths:\n"
    f"- Attachment 1 (baseline drawing): {MOUNT_SAMPLE}\n"
    f"- Attachment 2 (check drawing): {MOUNT_CHECK}\n\n"
    f"{user_prompt}\n"
    "Follow the steps in the missing-dimension-check skill.\n"
    "Tasks:\n"
    "2. Take inventory of the dimensions in the baseline drawing and list them.\n"
    "3. Split the check drawing into regions, visually compare them, and judge missing/mismatched values.\n"
    "4. Draw red circles + numbers at the missing positions on the check drawing and save the image under output/.\n"
    "5. Finally, report in English the list of missing dimensions, the list of value mismatches, the number-to-dimension legend, and a summary (missing count / total count),\n"
    "   and present the annotated image as an image in this chat.\n"
    "Do not fabricate numbers. If illegible, write 'Illegible'."
)

img_count = 0
with client.beta.sessions.events.stream(session.id) as stream:
    # Send the user message after opening the stream
    client.beta.sessions.events.send(
        session.id,
        events=[{"type": "user.message", "content": [{"type": "text", "text": instructions}]}],
    )

    for event in stream:
        match event.type:
            case "agent.message":
                for block in event.content:
                    print(block.text, end="")
            case "agent.tool_use":
                print(f"\n[tool: {getattr(event, 'name', event.type)}]")
            case "agent.tool_result":
                for block in (event.content or []):
                    if getattr(block, "type", None) == "image":
                        img_count += 1
                        save_image_block(block, img_count)
            case "session.status_idle":
                print("\n\n\u2705 Agent complete")
                break
            case "session.error":
                print(f"\n\u26A0\uFE0F Error: {event}")
                break

print(f"\nSaved images: {img_count} (./out)")

### Cleanup (optional)

Delete the session / agent / environment that are no longer needed (commented out to prevent accidental deletion). To delete the skill or the uploaded images, refer to the deletion cells in [02-diagram_check01.ipynb](02-diagram_check01.ipynb).

In [ ]:
# Cleanup (uncomment as needed to run)
# client.beta.sessions.delete(session.id)
# client.beta.agents.delete(agent.id)
# client.beta.environments.delete(environment.id)
# client.beta.files.delete(uploaded1.id)
# client.beta.files.delete(uploaded2.id)